In [3]:
import tensorflow as tf
import pandas as pd
import numpy as np
from tensorflow.keras.utils import Sequence
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, LSTM, GlobalAveragePooling2D, Concatenate, Dropout
from tensorflow.keras.applications import InceptionV3
from sklearn.model_selection import train_test_split

# Load handcrafted features
data = pd.read_csv("../Dataset/Features/signature_features.csv")
labels = data["label"].values
handcrafted_features = data.iloc[:, 2:].values  # Skip filename and label columns

# Image paths
image_size = (299, 299)  # InceptionV3 default input size
image_paths = [f"../Dataset/Processing/AugmentedDataset/{'Genuine' if label == 0 else 'Forged'}/{file}" 
               for file, label in zip(data["filename"], labels)]

# Split dataset
train_img_paths, val_img_paths, train_handcrafted, val_handcrafted, y_train, y_val = train_test_split(
    image_paths, handcrafted_features, labels, test_size=0.2, random_state=42, stratify=labels)

# Custom Data Generator
class SignatureDataGenerator(Sequence):
    def __init__(self, image_paths, handcrafted_features, labels, batch_size=32, shuffle=True):
        self.image_paths = image_paths
        self.handcrafted_features = handcrafted_features
        self.labels = labels
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.indexes = np.arange(len(self.image_paths))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.image_paths) / self.batch_size))  # Handle last batch

    def __getitem__(self, index):
        # Get batch indexes
        batch_indexes = self.indexes[index * self.batch_size:(index + 1) * self.batch_size]

        # Load batch data
        batch_images = np.array([self.load_image(self.image_paths[i]) for i in batch_indexes])
        batch_handcrafted = np.array([self.handcrafted_features[i] for i in batch_indexes])
        batch_labels = np.array([self.labels[i] for i in batch_indexes])

        return (batch_images, batch_handcrafted.reshape(-1, batch_handcrafted.shape[1], 1)), batch_labels

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indexes)

    @staticmethod
    def load_image(image_path):
        img = tf.keras.preprocessing.image.load_img(image_path, target_size=image_size)
        img = tf.keras.preprocessing.image.img_to_array(img)
        return tf.keras.applications.inception_v3.preprocess_input(img)

# Create data generators
batch_size = 32
train_generator = SignatureDataGenerator(train_img_paths, train_handcrafted, y_train, batch_size=batch_size)
val_generator = SignatureDataGenerator(val_img_paths, val_handcrafted, y_val, batch_size=batch_size)

# CNN Feature Extractor (InceptionV3)
inception_base = InceptionV3(weights=None, include_top=False, input_shape=(299, 299, 3))
cnn_output = GlobalAveragePooling2D()(inception_base.output)

# LSTM for sequence modeling
lstm_input = Input(shape=(train_handcrafted.shape[1], 1))  # Handcrafted features
lstm_layer = LSTM(128, return_sequences=False)(lstm_input)

# Combine CNN and LSTM
merged = Concatenate()([cnn_output, lstm_layer])
dense_layer = Dense(128, activation='relu')(merged)
dropout_layer = Dropout(0.5)(dense_layer)
output_layer = Dense(1, activation='sigmoid')(dropout_layer)

# Define hybrid model
model = Model(inputs=[inception_base.input, lstm_input], outputs=output_layer)

# Compile model
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# Train model using data generators
model.fit(train_generator, validation_data=val_generator, epochs=25)

# Save model
model.save("../Model/signature_forgery_detection_model.h5")

print("Model training completed and saved!")


Epoch 1/25


/Users/pasan_diksura/Documents/SoftwareDevelopment/Projects/Signature-Forgery-Detection-System/.venv/lib/python3.9/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


119/898 ━━━━━━━━━━━━━━━━━━━━ 38:06 3s/step - accuracy: 0.6013 - loss: 0.8118

KeyboardInterrupt: 